# Merge and Export Machine Learning Dataset
This notebook gathers all the pre-calculated features from the other scripts (OSM POIs, Land Use percentages, and Terrain Slope) and merges them into one single comprehensive `.csv` table that will be fed to the Machine Learning models.

## Note: Data Requirements
Currently, the other notebooks do not save their resulting pandas DataFrames to disk. For this notebook to effectively gather and merge them, we must first run the feature generation loops directly here, OR you must add an export block to the other notebooks (e.g. `dataset.to_csv("my_features.csv", index=False)`).

In this script, we'll write the logic that assumes those feature dataframes have either been saved to disk or can be reconstructed.

In [ ]:
import pandas as pd
import os

print("Libraries imported.")

Libraries imported.


### Load the datasets
**Assumption:** The datasets have been saved to a `../../data/processed/` folder from the previous notebooks. If not, go back to `UsosSol_percentages.ipynb`, `OSM_points.ipynb` and `Street_Slope copy.ipynb` and add `dataset.to_csv('../../data/processed/[name].csv', index=False)` at the very bottom of each.

In [ ]:
# Define file paths (UNCOMMENT AND ADJUST ONCE YOU HAVE EXPORTED THE DATA)

#path_landuse = "../../data/processed/landuse_features.csv"
path_osm_landuse = "../../data/processed/osm_landuse_features.csv"
path_slope = "../../data/processed/slope_features.csv"
path_bldg_height = "../../data/processed/osm_building_heights.csv"
path_lidar_height = "../../data/processed/lidar_building_heights.csv"
path_catastral_floors = "../../data/processed/catastral_building_floors.csv"
path_osm_points = "../../data/processed/osm_points_features.csv"
path_osm_roads = "../../data/processed/osm_roads_features.csv"

#df_landuse = pd.read_csv(path_landuse)
df_osm_landuse = pd.read_csv(path_osm_landuse)
df_slope = pd.read_csv(path_slope)
df_bldg_height = pd.read_csv(path_bldg_height)
df_lidar_height = pd.read_csv(path_lidar_height)
df_catastral_floors = pd.read_csv(path_catastral_floors)
df_osm_points = pd.read_csv(path_osm_points)
df_osm_roads = pd.read_csv(path_osm_roads)


print("Data loaded successfully.")

Data loaded successfully.
Data loaded successfully.


## Helo functions

In [59]:
# Help function to convert dB string to int

def extract_min(range_str):
    if range_str == '< 40 dB(A)':
        return 0
    return int((range_str).split(' - ')[0])

print(extract_min('70 - 75 dB(A)'))

70


In [ ]:
# Function to merge feature DataFrame with base DataFrame

def merge_feature_dataframe(base_df, feature_df, cols_to_drop):
    # Drop target columns
    clean_df = feature_df.drop(columns=[c for c in cols_to_drop if c in feature_df.columns], errors='ignore')
    
    # Drop duplicate columns (except street_id)
    cols_to_remove = [c for c in clean_df.columns if c in base_df.columns and c != 'street_id']
    clean_df = clean_df.drop(columns=cols_to_remove, errors='ignore')
    
    # Ensure street_id exists and is consistent
    if 'street_id' not in clean_df.columns or 'street_id' not in base_df.columns:
        return base_df
    
    clean_df['street_id'] = clean_df['street_id'].astype(str)
    base_df = base_df.copy()
    base_df['street_id'] = base_df['street_id'].astype(str)
    
    return base_df.merge(clean_df, on='street_id', how='left')

### Merge Datasets
We merge on the overarching street unique id (`TRAM`). Because we expect every street segment to be present in our base `noise` predictions, we will do a `LEFT` merge starting with the dataframe that contains the dependent noise variables.

In [60]:
ml_dataset = df_osm_roads.copy()

# Columns to remove to avoid repetition
cols_to_drop = ['noise_day', 'noise_evening', 'noise_night']

# Check street_id types before merging
print("Street ID types:")
print(f"ml_dataset: {ml_dataset['street_id'].dtype}")

for name, df in [
    ('OSM Land Use', df_osm_landuse),
    ('Slopes', df_slope),
    #('Building Heights', df_bldg_height),
    #('LiDAR Heights', df_lidar_height),
    ('Catastral Floors', df_catastral_floors),
    ('OSM Points', df_osm_points)
]:
    if 'street_id' in df.columns:
        print(f"{name}: {df['street_id'].dtype}")
    else:
        print(f"{name}: NO street_id column!")
    ml_dataset = merge_feature_dataframe(ml_dataset, df, cols_to_drop)

ml_dataset = ml_dataset.fillna(0)

ml_dataset['noise_day'] = ml_dataset['noise_day'].apply(extract_min)
ml_dataset['noise_evening'] = ml_dataset['noise_evening'].apply(extract_min)
ml_dataset['noise_night'] = ml_dataset['noise_night'].apply(extract_min)

display(ml_dataset.head(10))

print(f"Final dataset shape: {ml_dataset.shape}")

Street ID types:
ml_dataset: str
OSM Land Use: str
Slopes: str
Catastral Floors: str
OSM Points: str


,street_id,noise_day,noise_evening,noise_night,road_length,road_category,fid,osm_commercial_pct_50m,osm_green_pct_50m,osm_industrial_pct_50m,...,osm_residential_pct_50m,slope_pct,catastral_bldg_floors_mean_50m,catastral_bldg_floors_max_50m,catastral_bldg_floors_mean_100m,catastral_bldg_floors_max_100m,signal_count_50,poi_count_50,tree_count_50,transport_count_50
0,T04719W,70,65,60,48.661095,residential,0,0.0,1.809930,0.0,...,0.000000,0.904218,3.687500,9.0,3.198502,9.0,5,2,2,1
1,T19941Z,45,45,0,87.202693,living_street,1,0.0,10.295492,0.0,...,0.241591,0.493104,3.132353,8.0,3.811224,13.0,1,4,4,1
2,T18111R,55,55,50,79.934562,residential,2,0.0,7.545177,0.0,...,0.000000,1.738922,4.055556,10.0,3.840000,15.0,1,1,1,2
3,T03222Y,60,55,60,77.367711,living_street,3,0.0,1.706465,0.0,...,0.000000,5.751754,2.372449,10.0,2.623158,12.0,3,5,1,1
4,T17625I,55,55,50,107.679649,residential,4,0.0,0.000000,0.0,...,0.000000,3.575421,5.555556,10.0,4.391304,16.0,1,1,1,5
5,T05360P,55,55,50,91.759758,residential,5,0.0,0.000000,0.0,...,0.000000,1.318660,1.828571,5.0,2.187166,8.0,1,1,1,1
6,T08863T,50,50,45,86.869284,residential,6,0.0,0.000000,0.0,...,1.436572,3.131140,3.003759,11.0,2.748232,11.0,1,1,1,1
7,T00236S,45,45,40,57.158529,residential,7,0.0,0.000000,0.0,...,0.000000,0.174952,4.107296,10.0,3.924099,10.0,1,10,1,1
8,T13009A,55,55,50,147.700343,residential,8,0.0,0.000000,0.0,...,0.000000,4.989835,4.088889,11.0,3.824074,17.0,1,1,1,4
9,T11921P,60,55,50,61.576419,residential,9,0.0,8.395541,0.0,...,0.000000,6.317352,3.102041,8.0,3.081994,9.0,3,1,1,1


Final dataset shape: (15115, 21)


### Export to CSV
Finally, we dump out the master table into a standard format readable by `scikit-learn` in the next phase!

In [61]:
# UNCOMMENT TO EXPORT

output_dir = "../../data/machine_learning/"
os.makedirs(output_dir, exist_ok=True)

final_path = os.path.join(output_dir, "bcn_noise_ml_dataset.csv")
ml_dataset.to_csv(final_path, index=False)

print(f"Dataset successfully exported to: {final_path}")

Dataset successfully exported to: ../../data/machine_learning/bcn_noise_ml_dataset.csv
